# Lesson 52 · Audio Core complete — "show me the evidence": the tests all green, and the confidence of really understanding FFT → MFCC from scratch

**Goal**: get `tests/audio/` all green and clear Month 1 — **prove with evidence that you really understand it**.

> **A ceremony lesson**: checklist style — which modules you've written by hand, and how the tests pin down their behavior. **Don't pour in new concepts**; wherever something is fuzzy, go back to the matching lesson and fill the gap.

🔗 Aurora connection:
- `src/aurora/audio/io.py` — WAV reading and writing
- `src/aurora/audio/windows.py` — Hann / Hamming
- `src/aurora/audio/transforms.py` — DFT / FFT / IFFT / RFFT
- `src/aurora/audio/stft.py` — framing + the short-time Fourier transform
- `src/aurora/audio/mel.py` — Hz↔mel conversion, the mel filterbank, the mel spectrum
- `src/aurora/audio/mfcc.py` — DCT-II, MFCC

← **Previous lesson** · [L51 · MFCCs on real audio](L51_real_audio.ipynb)

> Last lesson covered **MFCCs on real audio**: extracting features from real WAV audio, with librosa as the answer key.
> This lesson explores **Audio Core complete**.

## This lesson's story: how do you prove you "really wrote" the FFT?

When someone presses you (an interviewer, a colleague, or your future self): "You say you implemented the FFT and MFCC from scratch — show me the evidence."

That question has two layers:
1. **The code layer**: does the test suite cover edge cases, numerical precision and shape correctness?
2. **The understanding layer**: without looking at the code, can you derive the whole chain of formulas on a whiteboard?

This section is the Audio Core's closing ceremony: get the 39 tests in `tests/audio/` all green, then distill the evidence that you "really understand" into a few points you can explain clearly in an interview.

**The complete chain of formulas from WAV to MFCC** (a 6-step mnemonic):

```
x[n]  → STFT: X[t,k] = Σ x[t·hop+n]·w[n]·e^{-2πikn/N}
     → power: P[t,k] = |X[t,k]|²
     → Mel: M[t,m] = Σ_k P[t,k]·H_m[k]        (triangular filters)
     → log: L[t,m] = log(M[t,m] + ε)
     → DCT: C[t,c] = Σ_m L[t,m]·cos(π·c·(2m+1)/(2N))·w(c)
     → truncate: MFCC[t, :n_mfcc]
```

**Three kinds of assertion in the test suite**:

| Assertion type | Example | What it means |
|---|---|---|
| Shape | `assert S.shape == (n_frames, n_bins)` | dimensions are right |
| Numerical | `np.allclose(my_fft, np.fft.fft, atol=1e-10)` | numerical precision |
| Property | `assert X[N-k] == conj(X[k])` | mathematical properties |

This section's goal: `make test` with zero failures, and the 5 key formulas worked into versions you can derive on a whiteboard and explain on the spot at any time.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import subprocess
import numpy as np

## 1. The complete pipeline from signal to features

```
WAV → framing (frame_signal) → windowing (hann/hamming) → FFT (rfft)
    → |·|² → mel filtering (mel_filterbank) → log → DCT-II → MFCC
```

The mathematical form of each step:

- **Framing**: cut a signal of length `T` into frames of length `N` with step `H`. Without padding there are `1 + (T-N)//H` frames; aurora defaults to `center=True` (first reflect-padding `N//2` at each end), and the frame count becomes `1 + T//H`.
- **Windowing**: multiply elementwise by `w[n] = 0.5 * (1 - cos(2π n / N))` (the periodic Hann window).
- **RFFT**: `X[k] = sum_{n=0}^{N-1} x[n] * e^{-j 2π k n / N}`, keeping the `N//2+1` non-redundant frequency bins.
- **Mel filtering**: `M[m] = sum_k fb[m,k] * |X[k]|²`, a triangular filter bank, with `fb` of shape `(n_mels, N//2+1)`.
- **Log compression**: `log_mel = 10 * log10(max(M, eps))`, imitating the logarithmic way the ear perceives loudness.
- **DCT-II (orthonormal)**: `C[i] = scale[i] * sum_m log_mel[m] * cos(π/M * (m+0.5) * i)`; the first 13 coefficients are the MFCCs.

### Key concepts in depth: why do these 6 steps pull features out of sound?

#### 🔑 Concept 1: the RFFT and conjugate symmetry — why is the frequency axis N//2+1 long?

You might ask: "If the signal is real, why is the FFT output still complex?"

**Intuitive explanation**: the Fourier transform asks "how much of frequency f is in this signal?". To answer that mathematically we need **complex numbers**, which describe two things at once:
- **magnitude**: how strong that frequency is
- **phase**: how far that frequency's wave is shifted relative to the reference time

Together, these two pieces of information are written as the complex number $X[k] = A[k] \cdot e^{j\phi[k]}$ (where $j = \sqrt{-1}$ is the imaginary unit; audio engineering writes $j$ rather than $i$ purely by convention).

**The beauty of the symmetry**: when the input signal is real (a microphone recording, say), the FFT output has a special property:
$$X[N-k] = \text{conj}(X[k])$$
that is, the complex values at "mirror frequencies" are conjugates. This means:
- frequencies 0 to N//2 completely determine the whole spectrum
- frequencies N//2+1 to N-1 just repeat information we already have

So the **RFFT** (real FFT) keeps only the first N//2+1 points, saving half the computation and storage.

**A numerical example**: with N=8,
```
DFT    outputs 8 complex numbers: [X₀, X₁, X₂, X₃, X₄, X₅, X₆, X₇]
RFFT   outputs 5 complex numbers: [X₀, X₁, X₂, X₃, X₄]  ← because 8//2+1=5
      the last 3 can be recovered by symmetry: X₅=conj(X₃), X₆=conj(X₂), X₇=conj(X₁)
```

#### 🔑 Concept 2: complex numbers and Euler's formula — why can a frequency be written as $e^{-j2\pi kn/N}$?

The DFT formula looks scary:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j 2\pi k n / N}$$

but all it's really answering is: "how much sinusoidal component at frequency k is there?"

**Euler's identity** tells us:
$$e^{j\theta} = \cos\theta + j\sin\theta$$

so:
$$e^{-j2\pi kn/N} = \cos(2\pi kn/N) - j\sin(2\pi kn/N)$$

which means the DFT is really **doing two things at once**:
- using $\cos$ to extract the amplitude of the cosine wave
- using $\sin$ to extract the amplitude of the sine wave (the phase difference)

Finally the two amplitudes combine into one complex number that records both strength and phase. That's why "representing frequency with complex numbers" is the neatest way to do it.

#### 🔑 Concept 3: the mel filter triangles — why go into the mel domain and back again?

So far what we have is a **Hz-domain spectrum** (an evenly spaced frequency axis). But the ear doesn't perceive frequency linearly:
- we can clearly tell 100 Hz from 200 Hz
- we can hardly hear the difference between 10,000 and 10,100 Hz

The **mel scale** is there to imitate the ear's logarithmic perception. The conversion formula (the HTK standard, which aurora uses by default) is:
$$\text{mel}(f) = 2595 \log_{10}(1 + f/700)$$

**Building the mel filters takes three steps**:

1. **Pick evenly spaced center points in the mel domain**
   Say we want 40 filters: take n_mels+2 = 42 evenly spaced points in the mel domain (including the two edges): $\text{mel}_{\min}, \text{mel}_{\min}+\Delta, \text{mel}_{\min}+2\Delta, ..., \text{mel}_{\max}$

2. **Transform back to the Hz domain**
   Use the inverse formula to turn those 42 mel points back into Hz:
   $$f = 700 \cdot (10^{\text{mel}/2595} - 1)$$
   Now they're uneven in the Hz domain — dense at low frequencies, sparse at high ones.

3. **Build the triangles**
   Each filter is a triangle:
   - left slope: rising from 0 to a peak of 1 (at the center point)
   - right slope: falling from 1 back to 0 (at the next center point to the right)

   As a formula, filter m is:
   $$H_m[k] = \begin{cases}
   0 & \text{if } k < f_m^{\text{left}} \\
   \frac{k - f_m^{\text{left}}}{f_m^{\text{center}} - f_m^{\text{left}}} & \text{if } f_m^{\text{left}} \le k < f_m^{\text{center}} \\
   \frac{f_m^{\text{right}} - k}{f_m^{\text{right}} - f_m^{\text{center}}} & \text{if } f_m^{\text{center}} \le k < f_m^{\text{right}} \\
   0 & \text{if } k \ge f_m^{\text{right}}
   \end{cases}$$

   where $f_m^{\text{left}}, f_m^{\text{center}}, f_m^{\text{right}}$ are the frequencies of the three corners.

What we end up with is an `(n_mels, n_fft//2+1)` matrix `fb`, each row one triangular filter.

#### 🔑 Concept 4: the DCT's orthonormalization — why sqrt(2/N)?

The basic DCT-II formula is:
$$C[i] = \sum_{m=0}^{M-1} x[m] \cdot \cos\left(\frac{\pi}{M}(m+0.5) i\right)$$

But the matrix this gives is not "orthogonal". What's an orthogonal matrix?

An **orthogonal matrix** satisfies $D \cdot D^T = I$ (the identity matrix). What does that buy us?
- energy conservation: if the DCT turns signal $x$ into $c$, then $\|x\|^2 = \|c\|^2$ (Parseval's theorem)
- well-behaved inversion: the inverse transform is very stable and doesn't easily amplify errors
- decorrelation: the DCT coefficients come out approximately decorrelated, which helps later machine learning

To make the matrix orthogonal we need weights:
$$D_{\text{orthonormal}}[i,m] = \sqrt{\frac{2}{M}} \cdot \cos\left(\frac{\pi}{M}(m+0.5) i\right) \quad (i > 0)$$
$$D_{\text{orthonormal}}[0,m] = \sqrt{\frac{1}{M}} \cdot \cos\left(\frac{\pi}{M}(m+0.5) \cdot 0\right) = \sqrt{\frac{1}{M}}$$

**Why is the first row special?** Because $\cos(0) = 1$: if it were also multiplied by $\sqrt{2/M}$, the first basis vector's energy would be twice the others', breaking orthonormality. So the first row uses $\sqrt{1/M}$.

#### 🔑 Concept 5: framing and center padding — why does the frame count go from 1+(T-N)//H to 1+T//H?

Without padding, cutting frames of length N from a signal of length T with step H:
- frame 0: positions 0 to N-1
- frame 1: positions H to H+N-1
- ...
- frame m: positions m·H to m·H+N-1 (the last frame needs m·H+N-1 < T)

Solving that inequality: $m \le \frac{T-N}{H}$, so at most $m_{\max} = \lfloor(T-N)/H\rfloor$, and the total frame count = $m_{\max}+1 = 1 + \lfloor(T-N)/H\rfloor$.

**Add center padding** (N//2 samples at each end):
- the padded signal's length becomes $T + 2 \cdot (N//2) \approx T + N$ (assuming N is even)
- the new frame count = $1 + \lfloor(T+N-N)/H\rfloor = 1 + \lfloor T/H \rfloor$

**Why pad?** So the start and end of the signal also get processed by a complete window, instead of wasting information because the window tapers to 0 at its ends.

**A numerical example**: sr=16000, dur=1s, n_fft=1024, hop=256
- T = 16000 (number of samples)
- no padding: $n_{\text{frames}} = 1 + \lfloor(16000-1024)/256\rfloor = 1 + 58 = 59$
- with padding (center=True): $n_{\text{frames}} = 1 + \lfloor 16000/256 \rfloor = 1 + 62 = 63$

**How exactly does "reflect padding" "fold"? — just saying "mirror" isn't precise enough**

You may still have a question at this point: what are the actual values of the `N//2` new padding samples? "Reflect" sounds like holding the end of the signal up to a mirror, but where exactly the mirror stands hides a detail that's easy to disagree on — **is the boundary sample itself copied too?**

An analogy: fold the end of a paper strip back on itself and press it onto the table. You can put "the crease right on the last point" (that point lies against its own mirror image, so it shows up twice), or "the crease halfway between the last point and the second-to-last" (the last point appears only once; it is itself the mirror). NumPy gives these two folds different names:

- `mode='symmetric'` (crease on a point; the boundary point is copied once)
- `mode='reflect'` (crease between two points; the boundary point itself isn't repeated, it appears only once)

A concrete example makes it clear: signal `x = [a, b, c, d, e, f, g, h]` (8 points), padded with 4 points on the left:

| Mode | Result (4 new points on the left + the original signal) |
|---|---|
| `mode='symmetric'` | `[d, c, b, a,  a, b, c, d, e, f, g, h]` — `a,b,c,d` flipped over as a block, **so a itself gets copied** |
| `mode='reflect'` | `[e, d, c, b,  a, b, c, d, e, f, g, h]` — `b,c,d,e` flipped over around `a`, **so a appears only once** (it's the mirror the crease runs through) |

(If what you guessed in your notes was `[e,d,c,b,a,b,c,d,...]` — congratulations, that's exactly the `reflect` answer!)

**Which one does aurora use?** One look at the line in the source `src/aurora/audio/stft.py` tells you:
```python
signal = np.pad(signal, pad, mode="reflect")
```
It's `reflect`, not `symmetric`. The right end works the same way: the second- to fifth-from-last points (`d,e,f,g`) are flipped around the last point `h` and appended after the signal.

The code cell below simply runs NumPy's two modes and prints the real output side by side — more reliable than memorizing a rule — and also takes a look at that line in aurora's source to confirm it really uses `reflect`.

In [ ]:
# Samuel's question: how exactly does reflect padding "fold"? A letter array makes the order crystal clear
x_letters = np.array(list("abcdefgh"))
print("original signal:                       ", x_letters)
print("mode='reflect'   (the one aurora uses):", np.pad(x_letters, 4, mode='reflect'))
print("mode='symmetric' (boundary repeats)   :", np.pad(x_letters, 4, mode='symmetric'))

# Read aurora's source directly to confirm it really uses reflect, rather than guessing
import inspect
from aurora.audio.stft import frame_signal

pad_line = [
    line.strip()
    for line in inspect.getsource(frame_signal).splitlines()
    if "np.pad" in line and "mode" in line
][0]
print(f"\nthe line actually called in aurora's stft.py: {pad_line}")
assert 'mode="reflect"' in pad_line
print("✅ confirmed: aurora's center padding uses reflect, not symmetric")

# Check again with a numeric signal — exactly the same pattern, but numbers make the fold positions easier to work out in your head
x_num = np.arange(1, 9)  # [1,2,3,4,5,6,7,8]
padded = np.pad(x_num, 4, mode='reflect')
print(f"\nnumeric check: {x_num} reflect-padded by 4 → {padded}")
print("the new [5,4,3,2] on the left is 2,3,4,5 flipped around the end point 1 (1 itself isn't repeated);")
print("the new [7,6,5,4] on the right is 4,5,6,7 flipped around the end point 8 (8 itself isn't repeated) —")
print("exactly the rule frame_signal(center=True) uses when it pads N//2 samples at each end.")

In [ ]:
from aurora.audio import sine, stft, mel_spectrogram, mfcc

sr = 16000
x = sine(440.0, 1.0, sr)          # 1 second of a pure 440 Hz tone

S = stft(x, n_fft=1024, hop_length=256)          # (n_frames, 513) complex
M = mel_spectrogram(x, sr, n_fft=1024, hop_length=256, n_mels=80)  # (n_frames, 80)
C = mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256)             # (n_frames, 13)

print(f"STFT  shape: {S.shape}  dtype: {S.dtype}")
print(f"Mel   shape: {M.shape}  dtype: {M.dtype}")
print(f"MFCC  shape: {C.shape}  dtype: {C.dtype}")
print(f"MFCC[0, :5] = {C[0, :5].round(4)}")
assert np.iscomplexobj(S)
assert M.shape[1] == 80
assert C.shape[1] == 13
print("✅ every pipeline shape is correct")

In [ ]:
# Numerical check: a concrete example of building the mel filter triangles
from aurora.audio.mel import hz_to_mel, mel_to_hz, mel_filterbank

# Example: 3 mel filters, n_fft=256 (a small value makes it easier to follow)
n_mels, n_fft, sr = 3, 256, 16000
fb_demo = mel_filterbank(n_mels=n_mels, n_fft=n_fft, sample_rate=sr)

n_freqs = n_fft // 2 + 1  # 129 frequency points
freq_hz = np.linspace(0, sr//2, n_freqs)  # 0, 62.5, 125, ..., 8000 Hz

print(f"Building mel filters, a demo (n_mels={n_mels}, n_fft={n_fft})")
print(f"total frequency points: {n_freqs}, range: 0 ~ {sr//2} Hz\n")

# Find each filter's three corners (its non-zero range)
for m in range(n_mels):
    nonzero_idx = np.where(fb_demo[m] > 0)[0]
    if len(nonzero_idx) > 0:
        left_idx, right_idx = nonzero_idx[0], nonzero_idx[-1]
        # Find the peak (the maximum)
        peak_idx = np.argmax(fb_demo[m])

        left_hz = freq_hz[left_idx]
        peak_hz = freq_hz[peak_idx]
        right_hz = freq_hz[right_idx]

        left_mel = hz_to_mel(left_hz) if left_hz > 0 else 0
        peak_mel = hz_to_mel(peak_hz)
        right_mel = hz_to_mel(right_hz)

        print(f"filter {m}:")
        print(f"  corners in Hz:  {left_hz:.1f} Hz → {peak_hz:.1f} Hz (peak) → {right_hz:.1f} Hz")
        print(f"  corners in mel: {left_mel:.1f} mel → {peak_mel:.1f} mel → {right_mel:.1f} mel")
        print(f"  note: close to evenly spaced in mel; in Hz, dense at low frequencies, sparse at high ✓\n")

# Check the RFFT symmetry (real signal)
x_real = np.sin(2 * np.pi * 440 * np.arange(1024) / 16000)
X_rfft = np.fft.rfft(x_real)
print(f"\n✅ RFFT symmetry check (N=1024):")
print(f"  RFFT output length: {len(X_rfft)} = 1024//2+1")
print(f"  input signal type: real")
print(f"  output type: complex ✓")

## 2. Checking the test suite

There are two test files under `tests/audio/` (in the [AURORA](https://github.com/djzoom/AURORA) repo, the reference library this course installs):

| File | Coverage | Key assertions |
|------|----------|----------|
| `test_transforms.py` | DFT / FFT / IFFT / RFFT | `atol=1e-9` against `numpy.fft` |
| `test_features.py` | windows / STFT / mel / WAV I/O | `atol=1e-12` for window functions / DCT-II |
| `test_features.py` (MFCC) | MFCC shape only | shape assertion only, `C.shape[1] == 13`; **no numerical atol test yet** |

Some key test points:

```python
# FFT agrees with NumPy element by element
np.testing.assert_allclose(T.fft(x), np.fft.fft(x), atol=1e-9)

# DCT-II by the hand-written formula
basis = np.cos(np.pi / n * np.outer(k, k + 0.5))
ref = x @ basis.T * scale
np.testing.assert_allclose(dct_ii(x), ref, atol=1e-12)

# Hann window agrees with numpy.hanning (symmetric mode)
np.testing.assert_allclose(hann(64, periodic=False), np.hanning(64), atol=1e-12)
```

The design principle behind these tests: every function has a reference answer that is independent of the implementation (a formula or NumPy), so as long as the maths is right, the test is bound to pass.

In [ ]:
import sys
from pathlib import Path
import aurora

# tests/audio/ lives in the AURORA repo (the reference library), not in this English edition:
# look for it next to the installed aurora package (a clone installed with `pip install -e .`),
# then in the current git repo.
_candidates = [Path(aurora.__file__).resolve().parents[2]]
try:
    _candidates.append(Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True, stderr=subprocess.DEVNULL).strip()))
except Exception:
    pass
aurora_root = next((p for p in _candidates if (p / "tests" / "audio").is_dir()), None)
assert aurora_root is not None, (
    "❌ tests/audio/ not found: clone https://github.com/djzoom/AURORA, "
    "run `pip install -e .` inside it, then rerun this cell"
)

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/audio/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
    cwd=aurora_root,  # aurora repo root
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "❌ some tests failed; check the output above"
print("✅ tests/audio/ all passed")

## 3. The Month 1 delivery standard

Three layers of verification to clear it:

**Layer 1 — the code runs**: `make test` with zero failures, and `make demo` outputs the correct spectrogram.

**Layer 2 — you can explain the maths**: you can rewrite any one of the functions in front of a blank file, line by line against the formula, without relying on memory:
- `dct_ii`: `basis[k, m] = cos(π/N * (m+0.5) * k)`, orthonormalization factor `sqrt(2/N)`, first row `sqrt(1/N)`.
- `mel_filterbank`: the triangle corners are evenly spaced in the mel domain, and uneven once converted back to Hz.
- `fft`: the Cooley–Tukey butterfly decomposition, bringing `O(N²)` down to `O(N log N)` when `N=2^k`.

**Layer 3 — you can explain the differences**: you can explain why aurora's and librosa's outputs differ slightly (normalization strategy, mel-scale parameters, center-padding behavior).

```
Checklist
[ ] pytest tests/audio/  →  all green
[ ] can write dct_ii(x) by hand and explain every line
[ ] can write mel_filterbank(n_mels, n_fft, sr) by hand and explain the triangle corner coordinates
[ ] can explain the difference between hann(N, periodic=True) and periodic=False
[ ] can say where the stft output shape (n_frames, N//2+1) comes from
```

In [ ]:
from aurora.audio import (
    sine, hann, hamming, dct_ii, mel_filterbank,
    frame_signal, stft, mel_spectrogram, mfcc,
    hz_to_mel, mel_to_hz, read_wav, write_wav
)

# Check 1: DCT-II verified by hand
x_test = np.array([1.0, 2.0, 3.0, 4.0])
n = len(x_test)
k = np.arange(n)
basis = np.cos(np.pi / n * np.outer(k, k + 0.5))
scale = np.full(n, np.sqrt(2.0 / n)); scale[0] = np.sqrt(1.0 / n)
ref_dct = x_test @ basis.T * scale
np.testing.assert_allclose(dct_ii(x_test), ref_dct, atol=1e-12)
print("✅ DCT-II hand-written formula check passed")

# Check 2: mel filterbank shape and non-negativity
fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)
assert fb.shape == (40, 513) and np.all(fb >= 0.0)
print(f"✅ mel_filterbank shape={fb.shape}, all non-negative, every row peaks > 0: {np.all(fb.max(axis=1) > 0)}")

# Check 3: Hann window, periodic vs symmetric
per = hann(8, periodic=True)
sym = hann(9, periodic=False)
np.testing.assert_allclose(per, sym[:-1], atol=1e-12)
print("✅ hann periodic=True equals symmetric N+1 with the last point dropped")

# Check 4: STFT shape
x_1s = sine(440.0, 1.0, 16000)
S = stft(x_1s, n_fft=1024, hop_length=256)
assert S.shape[1] == 1024 // 2 + 1
print(f"✅ stft output {S.shape}, frequency axis = N//2+1 = {1024//2+1}")

# Check 5: the mel conversion is invertible
freqs = np.array([0.0, 100.0, 440.0, 1000.0, 8000.0])
np.testing.assert_allclose(mel_to_hz(hz_to_mel(freqs)), freqs, atol=1e-6)
print("✅ hz_to_mel → mel_to_hz round-trip error < 1e-6 Hz")

## Parameter experiment: aurora.mfcc vs librosa.feature.mfcc

**Parameters**: `n_fft=1024, hop_length=256, n_mels=80, n_mfcc=13`; the signal is 1 second of a pure 440 Hz tone.

**What to expect**:
- the same shape `(n_frames, 13)` (both default to `center=True`, so the frame counts agree), with values close but not identical.
- the main sources of difference:
  1. mel scale and normalization: aurora uses the HTK formula `2595 * log10(1 + f/700)` with unnormalized triangular filters; librosa defaults to the **Slaney** scale (`htk=False`) plus `norm="slaney"` area normalization — this is the main source of the numerical difference.
  2. padding mode: both pad `n_fft//2` at the start and end, but aurora uses reflect padding while recent librosa versions default to zero padding; this only affects the first and last few frames.
  3. the power parameter: librosa builds the mel spectrum from the power spectrum (`|X|²`) by default, and aurora's `mel_spectrogram` is also a power spectrum, so on this point they agree.

Run the code cell below and look at the order of magnitude of the RMSE and the difference in frame 0.

## Common confusions: FAQ

### Q1: what is the eps in "log compression" for? Why might M[t,m] contain zeros?

**Question**: the code says `log_mel = 10 * log10(max(M, eps))`. Why the eps? Does M have zeros?

**Answer**: yes, M can contain zeros. The reason is simple — the mel spectrum is `M[t,m] = sum_k P[t,k] * H_m[k]`, and if some mel band has no energy at all in some frame (say that stretch of sound is too quiet), M[t,m] is 0.

Then you take the log of 0: `log10(0) = -∞`, and the numbers blow up. So you must first give it a tiny positive floor eps (usually 1e-10): `max(M, eps)` lifts the 0 up to eps, which turns it into `log10(eps)` ≈ -10, a finite large negative number that later steps can handle. (Note that eps is a floor taken with max, not something added on — L50 explained that the two aren't equivalent, and `power_to_db` uses `max(M, amin)`.)

This kind of treatment is called **dynamic range compression** — using a log to squeeze a huge numerical range (1e8 down to 1) into a manageable one (80 down to 0).

### Q2: what is `np.outer()`, and how does it relate to matrix multiplication?

**Question**: the DCT check code uses `basis = np.cos(np.pi * np.outer(k, 2*n_arr+1) / (2*N_dct))`. What is np.outer()?

**Answer**: `np.outer(a, b)` computes the outer product of two 1-D arrays.

```python
a = np.array([1, 2, 3])        # shape (3,)
b = np.array([10, 20])         # shape (2,)
np.outer(a, b) = [
  [1*10,  1*20],
  [2*10,  2*20],
  [3*10,  3*20]
]  # shape (3, 2)
```

**How it relates to matrix multiplication**: if a is a column vector (shape (m,1)) and b is a row vector (shape (1,n)), then `a @ b` equals `np.outer(a, b)`.

In the DCT we need the product of every basis index k with every input point n, forming a 2-D matrix. `np.outer()` is the neatest way to write it.

### Q3: "periodic" vs "symmetric" Hann windows — when do you use which?

**Question**: the code says `hann(8, periodic=True) ≈ hann(9, periodic=False)[:-1]`. What's the difference between the two?

**Answer**: the two windows' formulas are:
- **symmetric** window: $w[n] = 0.5 \cdot (1 - \cos(2\pi n / (N-1)))$, where N is the window length; both ends come symmetrically close to 0
- **periodic** window: $w[n] = 0.5 \cdot (1 - \cos(2\pi n / N))$, treating N as the period

Mathematically, `hann(8, periodic=True)` equals taking the first 8 points of `hann(9, periodic=False)`.

**When to use which?**
- **The STFT uses the periodic window** (`periodic=True`): STFT frames slide along, and consecutive frames overlap by 75% (with the usual hop=N/4). The periodic window keeps "energy conservation" across frame boundaries (the COLA condition), so no audio is lost and nothing jumps.
- **The symmetric window** (`periodic=False`) is mainly the convention in FIR filter design; even for a one-off FFT of one stretch of signal for spectral analysis, the periodic window is usually recommended, and for large N the two barely differ.

Aurora's STFT uses `periodic=True` by default, so the energy of overlapping frames adds up correctly.

### Q4: Slaney vs HTK — why do librosa and aurora differ?

**Question**: the code says aurora uses HTK and librosa defaults to Slaney. What are these?

**Answer**: they are two different mel-scale standards, defined by different research groups:

| Standard | mel formula | normalization | used in |
|------|----------|--------|------|
| **HTK** | `2595 * log10(1 + f/700)` | unnormalized | academia and speech recognition |
| **Slaney** | linear below 1 kHz, logarithmic above (not the same formula) | **area normalization** | librosa default |

Besides the formula, the other key difference is **area normalization**. What does that mean?

The area of each triangular filter (∫ H[f] df) naturally differs because the frequency spacing differs. The Slaney standard weights each filter's coefficients so that every filter has the same area. The benefit: acoustic features in different frequency bands are "treated fairly".

**The effect in practice**:
- with HTK (aurora): the mel spectrum looks stronger at high frequencies
- with Slaney (librosa): the mel spectrum is more balanced across the whole range

Which to choose? If you're matching a paper, use what the paper uses. For your own project, Slaney's numerical balance is better.

### Q5: why keep only the first 13 MFCC coefficients at the end?

**Question**: the DCT outputs M dimensions (M = number of mel filters, usually 40), but only the first 13 are used. Why?

**Answer**: it's an **engineering convention** with some history behind it:

1. **Energy fall-off**: the later DCT coefficients represent fine high-order detail of the mel spectrum. For speech recognition, that detail is mostly noise and little signal — not worth it.
2. **Research findings**: speech-recognition research in the 1980s found that the first 12–13 MFCC coefficients already hold most of the information useful for recognition.
3. **Standardization**: it later became an industry standard; most audio systems use 13 (or 13 + 13 deltas = 26).

Modern deep learning sometimes uses more (say 40), because neural networks make better use of the high-order information. But in classical systems, 13 is the most economical choice.

### Q6: why do the tests use `atol=1e-9` rather than exact equality?

**Question**: test_transforms.py says `atol=1e-9`. Why can't it be 0?

**Answer**: floating-point precision is limited. Computers use 64-bit floating point (double precision), with a machine epsilon of about 2.22e-16.

An FFT does N·log₂(N) multiply-adds. Each one has a rounding error, and the errors accumulate:
- for N=1024, log₂(N)=10, so about 10240 operations
- accumulated error ≈ 2.22e-16 × 10240 ≈ 2.3e-12

So setting `atol=1e-9` leaves enough "tolerance" to catch real bugs (a wrong formula, say) without failing on floating-point rounding.

Set atol too strict (say 1e-15) and even perfectly correct code fails because of rounding; set it too loose (say 1e-6) and real bugs slip through.

In [ ]:
try:
    import librosa
    HAS_LIBROSA = True
except ImportError:
    HAS_LIBROSA = False
    print("librosa not installed; skipping the comparison (pip install librosa and try again)")

if HAS_LIBROSA:
    from aurora.audio import sine, mfcc as aurora_mfcc

    sr = 16000
    x = sine(440.0, 1.0, sr).astype(np.float32)

    # aurora (center=True, reflect padding)
    C_aurora = aurora_mfcc(x.astype(np.float64), sr,
                           n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80)

    # librosa (center=True, zero padding by default)
    C_lib_center = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80
    ).T  # librosa returns (n_mfcc, n_frames); transpose to (n_frames, n_mfcc)

    print(f"aurora  MFCC shape: {C_aurora.shape}")
    print(f"librosa MFCC shape: {C_lib_center.shape}")

    # Compare after aligning the frame counts
    n_common = min(C_aurora.shape[0], C_lib_center.shape[0])
    rmse = np.sqrt(np.mean((C_aurora[:n_common] - C_lib_center[:n_common])**2))
    print(f"RMSE over the first {n_common} frames = {rmse:.4f}")
    print(f"aurora  frame 0, first 5 coefficients: {C_aurora[0, :5].round(3)}")
    print(f"librosa frame 0, first 5 coefficients: {C_lib_center[0, :5].round(3)}")
    print()
    print("Where the differences come from:")
    print(f"  frame-count difference = {C_lib_center.shape[0] - C_aurora.shape[0]} (both use center=True, so it should be 0)")
    print("  the numerical difference comes mainly from the mel scale: aurora uses HTK, unnormalized; librosa defaults to Slaney + norm='slaney'")

## Appendix · Audio Core graduation: the abilities you now have

- [ ] can rewrite `fft` / `stft` / `mel_filterbank` / `mfcc` from a blank page (spoken + whiteboard)
- [ ] `make test` audio all green
- [ ] can explain the MFCC shapes along the way: `(T, n_mels)` → `(T, 13)`
- [ ] **the next step is not more DSP** but **L54: learning how a model learns from new data**

> This checklist isn't extra homework; it's how Month 1 wraps up: once you can explain all four out loud, switching over to deep learning won't leave a gap.

## Wrap-up

`mfcc()`, `mel_filterbank()`, `dct_ii()` and `stft()` make up the complete feature-extraction stack of the Aurora Audio Analysis Engine; every function has passed the numerical assertions in `tests/audio/` (with `atol` as strict as `1e-12`), confirming agreement with the NumPy / formula references.
Month 1's deliverable is a **pure-NumPy audio feature pipeline with zero black boxes**, covering every step from WAV I/O to 13-dimensional MFCC vectors.
The `aurora.audio` module can now serve as the feature-extraction backend for the deep-learning modules that follow (L54+), calling `mfcc()` directly to output an `(n_frames, 13)` tensor.
The next lesson (L53) turns these numbers into pictures: waveform → spectrogram → mel spectrum → cepstral coefficients, visualized layer by layer.

## ✏️ Whiteboard challenge: six Audio Core interview questions in a row (target: 15 minutes)

Cover the screen and derive on paper:

**Question 1**: write down the DFT formula and explain how the FFT brings the complexity from O(N²) down to O(N·logN).
(divide and conquer: X[k]=E[k]+W^k·O[k]; log₂N levels; N/2 butterflies per level)

**Question 2**: what is the Hann window formula? Why does it reduce spectral leakage?
(w[n]=0.5(1-cos(2πn/N)); it tapers to 0 at both ends, removing the jump at the boundary)

**Question 3**: the three steps for building the mel filterbank:
① evenly spaced center points in the mel domain → ② transform back to Hz → ③ build the triangles → matrix shape?

**Question 4**: what are the normalization factors w(k) of the orthonormal DCT-II?
(k=0: 1/√N, k>0: √(2/N); satisfying D·Dᵀ=I)

**Question 5**: the shape flow through the full MFCC pipeline (fill in the numbers):
`sr=16000, n_fft=1024, hop=256, dur=1s, n_mels=40, n_mfcc=13`
→ n_frames=?, and each step's output shape?

**Question 6**: why do the tests use `atol=1e-9` rather than `atol=0`?
(accumulated floating-point error; the FFT's log₂N levels give an error ∝ N·ε_machine)

Once you've worked it out, run the cell below to check the numbers.

### Question 2: how the Hann window reduces spectral leakage (why are the boundaries so important?)

**What spectral leakage is**: when you take the FFT of a finite stretch of signal, its two ends usually don't join up (there's a jump). That jump "spreads out" in the frequency domain, which is called spectral leakage.

**Example**:
- no window: take the FFT of the rectangular-windowed signal directly
- with a Hann window: both ends taper to 0, and the boundaries are smooth

**The mathematical reason**:
Windowing means multiplying the signal x[n] by the window function w[n]. The key property of the Hann window is:
$$w[0] = w[N-1] = 0$$

This guarantees the windowed signal $x[n] \cdot w[n]$ is 0 at both ends, with no jump. The rectangular window, by contrast, jumps at the boundaries, and its sinc sidelobes "leak" into neighboring frequencies.

**The numerical consequences of leakage**:
- without a window, the peak sidelobe in the frequency domain reaches -13 dB
- with a Hann window, leakage drops to -32 dB (19 dB lower)

**When do you care about spectral leakage?**
- **in the STFT**: many overlapping frames; the window reduces leakage so each frame's energy doesn't get muddled
- **in a one-off FFT analysis**: when you need to pinpoint a narrowband frequency component precisely, you must use a Hann or Hamming window

**The two numbers "-13 dB vs -32 dB" alone are hard to picture — so let's just plot them**

Ruth put it well in her notes: "This involves an important property of the Fourier transform, but you need to look at the plots in the frequency domain to really understand it." Numbers really are abstract, but plots don't lie. The next code cell draws 4 plots:

- **top left**: the rectangular-window (no window) waveform in time — notice that at the two ends (n=0 and n=255) the signal is not 0 and the ends don't match up; that's the "boundary jump"
- **top right**: the Hann-windowed waveform in time — both ends are pressed down to 0, the transition is smooth, no jump
- **bottom left**: the rectangular window's spectrum (in dB) — the main lobe (the spike in the middle) is narrow, but the sidelobes on either side are high and trail far out; that's "leakage"
- **bottom right**: the Hann window's spectrum (in dB) — the main lobe is a little wider (that's the price), but the sidelobes are pushed much lower and much cleaner

**The key to reading the plots**: the main lobe is where the signal's true frequency is; the sidelobes are "leaked" spurious energy that shouldn't be there at all — artifacts created by the boundary jump. Choosing a window is a trade-off between "main-lobe width vs sidelobe height" — Hann trades a slightly wider main lobe for much lower sidelobes, and for most audio-analysis tasks that's a very good deal.

### Question 4, more: what orthogonality of the DCT means intuitively

**What an orthogonal matrix is**:

A matrix D is orthogonal if and only if $D \cdot D^T = I$ (the identity matrix).

Intuitively:
- every row of D is a vector of length 1 (a unit vector)
- any two rows of D are perpendicular (orthogonal) to each other

**What it buys us**:
1. **Energy conservation**: if you project the signal x with D (i.e. $c = D \cdot x$), then $\|c\|^2 = \|x\|^2$
2. **Stable inversion**: $D^{-1} = D^T$ — extremely simple and stable, unlike general matrix inversion, which easily amplifies errors
3. **Decorrelation**: the DCT coefficients come out approximately decorrelated, so later machine-learning models find it easier to learn meaningful patterns

**What this means for the DCT-II**:

The unweighted DCT basis vectors are:
$$d_k[m] = \cos\left(\frac{\pi}{M}(m+0.5)k\right)$$

and their "energies" (squared 2-norms) aren't equal:
- for k=0: $\|d_0\|^2 = M$ (all ones)
- for k>0: $\|d_k\|^2 = M/2$ (because of the cosine's symmetry)

So the weights correct this to:
$$d_k^{\text{norm}}[m] = \sqrt{\frac{1}{M}} \cdot \cos(...) \quad \text{(k=0)}$$
$$d_k^{\text{norm}}[m] = \sqrt{\frac{2}{M}} \cdot \cos(...) \quad \text{(k>0)}$$

Now every basis vector has unit length, and the conditions for an orthogonal matrix are automatically met.

### Question 5, more: working out the shape flow concretely

**Deriving the frame count from a whole number of seconds**:

Parameters: sr=16000, duration=1s, n_fft=1024, hop_length=256, n_mels=40

1. **Total samples**: N = sr × duration = 16,000 samples

2. **The center=True padding**:
   - n_fft//2 = 512 samples padded at each end (reflect padding mode)
   - padded length = 16,000 + 2×512 = 17,024

3. **Number of frames**:
   - each frame is 1024 long, with a step of 256
   - frames = 1 + ⌊(17,024 - 1024) / 256⌋ = 1 + ⌊62.5⌋ = 1 + 62 = **63 frames**

4. **Each step's shape**:
   - raw signal: (16000,)
   - STFT output: (63, 513) = (n_frames, n_fft//2+1)
   - mel spectrum: (63, 40) = (n_frames, n_mels)
   - MFCC: (63, 13) = (n_frames, n_mfcc)

**Why does center change the frame count?**

Without padding, the last frame's right edge has to stay inside the signal:
- frame start + frame length - 1 < N
- m × H + N_fft - 1 < N
- m < (N - N_fft) / H
- m_max = ⌊(16000 - 1024) / 256⌋ = 58, 59 frames in all

After padding:
- m < (N_padded - N_fft) / H = (17024 - 1024) / 256 = 62.5
- m_max = 62, 63 frames in all

**Check**: center padding adds 4 frames (from 59 to 63). This ensures the start and end of the signal aren't "eaten" by the zeros at the two ends of the window.

In [ ]:
import matplotlib.pyplot as plt
from aurora.audio.windows import hann

# Ruth's question: the two numbers "-13dB vs -32dB" alone aren't intuitive enough, so plot the spectral leakage directly
N = 256
n = np.arange(N)
f0 = 10.5  # deliberately a non-integer number of cycles: one cycle takes N/f0 samples, not a whole number, so the ends don't meet
sig = np.sin(2 * np.pi * f0 * n / N)

w = hann(N)
sig_rect = sig * 1.0   # "no window" = multiplying by an all-ones rectangular window
sig_hann = sig * w     # with a Hann window

# Zero-padding to a longer FFT in the frequency domain only makes the curves smoother and easier to see;
# it doesn't change the width of the window itself (the true frequency resolution is still set by N=256)
NFFT = 4096
rect_spec = np.abs(np.fft.rfft(sig_rect, n=NFFT))
hann_spec = np.abs(np.fft.rfft(sig_hann, n=NFFT))
rect_db = 20 * np.log10(rect_spec / rect_spec.max() + 1e-12)
hann_db = 20 * np.log10(hann_spec / hann_spec.max() + 1e-12)
freq_bins = np.fft.rfftfreq(NFFT) * N  # rescaled back to "original bin number" units, to compare with the formula

fig, axes = plt.subplots(2, 2, figsize=(11, 6))

axes[0, 0].plot(n, sig_rect)
axes[0, 0].set_title('Rectangular (no) window, time domain: ends don\'t match → boundary jump')
axes[0, 0].set_xlabel('sample n')

axes[0, 1].plot(n, sig_hann, color='C1')
axes[0, 1].set_title('Hann window, time domain: ends pressed to 0 → smooth, no jump')
axes[0, 1].set_xlabel('sample n')

axes[1, 0].plot(freq_bins, rect_db)
axes[1, 0].set_xlim(0, 40); axes[1, 0].set_ylim(-80, 5)
axes[1, 0].set_title('Rectangular window spectrum (dB): narrow main lobe, high far-reaching sidelobes = leakage')
axes[1, 0].set_xlabel('frequency bin'); axes[1, 0].set_ylabel('dB (relative to main-lobe peak)')
axes[1, 0].grid(True, alpha=.3)

axes[1, 1].plot(freq_bins, hann_db, color='C1')
axes[1, 1].set_xlim(0, 40); axes[1, 1].set_ylim(-80, 5)
axes[1, 1].set_title('Hann window spectrum (dB): wider main lobe, but much cleaner sidelobes')
axes[1, 1].set_xlabel('frequency bin'); axes[1, 1].set_ylabel('dB (relative to main-lobe peak)')
axes[1, 1].grid(True, alpha=.3)

plt.tight_layout()
plt.show()


# Put numbers on the difference your eyes see in the two plots: how much lower are the sidelobes? How much wider is the main lobe?
def sidelobe_db(db, main_bin, guard):
    mask = np.ones_like(db, dtype=bool)
    lo, hi = max(0, main_bin - guard), min(len(db), main_bin + guard + 1)
    mask[lo:hi] = False
    return db[mask].max()


def mainlobe_width_bins(db, main_bin, thresh=-3):
    lo = main_bin
    while lo > 0 and db[lo] > thresh:
        lo -= 1
    hi = main_bin
    while hi < len(db) - 1 and db[hi] > thresh:
        hi += 1
    return hi - lo


main_rect = int(np.argmax(rect_spec))
main_hann = int(np.argmax(hann_spec))
guard = NFFT // N  # after zero-padding, one "original bin" is this many new bins wide
sl_rect = sidelobe_db(rect_db, main_rect, guard * 2)
sl_hann = sidelobe_db(hann_db, main_hann, guard * 2)
mw_rect = mainlobe_width_bins(rect_db, main_rect) / guard
mw_hann = mainlobe_width_bins(hann_db, main_hann) / guard

print(f"Measured in this example (N={N}):")
print(f"  rectangular window: highest sidelobe = {sl_rect:.1f} dB, main-lobe width (-3dB) ≈ {mw_rect:.1f} original bins")
print(f"  Hann window: highest sidelobe = {sl_hann:.1f} dB, main-lobe width (-3dB) ≈ {mw_hann:.1f} original bins")
print(f"  sidelobes lowered by {sl_rect - sl_hann:.1f} dB — the same order of magnitude as the '-13dB → -32dB' quoted earlier")
print(f"  main lobe about {mw_hann/mw_rect:.1f}× wider — that's the Hann window's price: coarser resolution in exchange for clean sidelobes")

## The six interview questions: step-by-step derivations

Before the whiteboard challenge, let's see how to derive things step by step. Detailed steps are given here for Questions 1 and 3; for the others, follow the same line of thinking.

### Question 1: why the FFT is fast (from O(N²) to O(N·logN))

**DFT definition**:
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j 2\pi k n / N}, \quad k = 0, 1, ..., N-1$$

**Naive computation**: each k takes N multiplications, and there are N values of k, so N² operations.

**Cooley–Tukey divide and conquer** (assuming N = 2^m):

Split the input into even and odd terms:
$$X[k] = \sum_{n \text{ even}} x[n] e^{-j2\pi kn/N} + \sum_{n \text{ odd}} x[n] e^{-j2\pi kn/N}$$

Let $n = 2l$ (even) and $n = 2l+1$ (odd), where $l = 0, 1, ..., N/2-1$:

$$X[k] = \sum_{l=0}^{N/2-1} x[2l] e^{-j2\pi k(2l)/N} + \sum_{l=0}^{N/2-1} x[2l+1] e^{-j2\pi k(2l+1)/N}$$

$$= \sum_{l=0}^{N/2-1} x[2l] e^{-j2\pi kl/(N/2)} + e^{-j2\pi k/N} \sum_{l=0}^{N/2-1} x[2l+1] e^{-j2\pi kl/(N/2)}$$

$$= E[k] + W^k \cdot O[k]$$

where:
- $E[k]$ = the DFT of the even terms (length N/2)
- $O[k]$ = the DFT of the odd terms (length N/2)
- $W^k = e^{-j2\pi k/N}$ is called the "twiddle factor"

**Complexity analysis**:
- computing E and O: (N/2)² operations each = 2 × (N/2)² = N²/2
- merging E and O: N multiplications
- total: N²/2 + N ≈ N²/2 (for large enough N)

**Recursing further** (level 1, taking N=1024):
- 2 DFT-512s → 2 × 512²/2 = 512²
- plus merging → +512
- total: 512² + 512

Keep splitting until the pieces have length 1: there are log₂(N) levels, and the merges at each level cost about N operations in total, so:
$$T(N) = N \log_2 N$$

**Concrete count for N=1024**:
- naive DFT: 1024² = 1,048,576 operations
- FFT: 1024 × 10 = 10,240 operations (plus the overhead of the butterfly merges)
- **speedup ≈ 100×**

### Question 3: the three steps for building the mel filters (with real numbers)

Parameters: n_mels=3, n_fft=512, sr=16000

**Step 1**: evenly spaced center frequencies in the mel domain

The mel frequency range: from 0 Hz to the Nyquist frequency (sr/2 = 8000 Hz)

$$\text{mel}_{\min} = \text{hz2mel}(0) = 0$$
$$\text{mel}_{\max} = \text{hz2mel}(8000) = 2595 \log_{10}(1 + 8000/700) \approx 2840 \text{ mel}$$

Split evenly into 3+2=5 points (n_mels + 2 boundary points):
$$\text{mel}_i = 0 + i \cdot \frac{2840}{4}, \quad i=0,1,2,3,4$$
$$= 0, 710, 1420, 2130, 2840 \text{ mel}$$

**Step 2**: transform back to Hz

Using the inverse formula: $f = 700(10^{\text{mel}/2595} - 1)$

$$f_0 = 700(10^{0} - 1) = 0 \text{ Hz}$$
$$f_1 = 700(10^{710/2595} - 1) \approx 700(10^{0.274} - 1) \approx 700 \times 0.878 \approx 614 \text{ Hz}$$
$$f_2 = 700(10^{1420/2595} - 1) \approx 1768 \text{ Hz}$$
$$f_3 = 700(10^{2130/2595} - 1) \approx 3934 \text{ Hz}$$
$$f_4 = 700(10^{2840/2595} - 1) \approx 8000 \text{ Hz}$$

Now we have 5 frequency points in Hz (uneven): 0, 614, 1768, 3934, 8000 Hz

**Step 3**: build the triangles

- **Filter 0**: corners at f₀=0, f₁=614, f₂=1768 Hz
  - 0 to 614 Hz: rising linearly (slope = 1/(614-0))
  - 614 to 1768 Hz: falling linearly (slope = -1/(1768-614))
  - everywhere else: 0

- **Filter 1**: corners at f₁=614, f₂=1768, f₃=3934 Hz

- **Filter 2**: corners at f₂=1768, f₃=3934, f₄=8000 Hz

**Observations**:
- evenly spaced in the mel domain (710 mel each)
- in Hz the spacing keeps growing (614, 1154, 2166, 4066 Hz) ✓ exactly what we want
- each triangle's "width" (its span in Hz) grows with frequency

In [ ]:
# ✏️ Answer-check cell for the six interview questions
import numpy as np, sys
sys.path.insert(0, 'src')

# Q1: FFT complexity — counting butterflies
N = 8
n_layers = int(np.log2(N))
butterflies = N // 2 * n_layers
dft_ops = N * N
speedup = dft_ops / butterflies
assert n_layers == 3 and butterflies == 12 and speedup > 5
print(f"Q1 ✅  N={N}: DFT {dft_ops} ops, FFT {butterflies} butterflies, {speedup:.1f}× faster")
# The bigger N, the bigger the advantage: at N=1024 the speedup is ~205×
N_big = 1024
speedup_big = N_big**2 / (N_big // 2 * int(np.log2(N_big)))
assert speedup_big > 100
print(f"      N={N_big}: DFT {N_big**2} ops vs FFT {N_big//2*int(np.log2(N_big))} butterflies, {speedup_big:.0f}× faster")

# Q2: Hann window boundary = 0
from aurora.audio.windows import hann
w = hann(16)
assert np.isclose(w[0], 0.0, atol=1e-12)
print(f"Q2 ✅  Hann window w[0]={w[0]:.1f} (boundary = 0, removing the jump)")

# Q3: mel filterbank shape
from aurora.audio.mel import mel_filterbank
fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)
assert fb.shape == (40, 513) and np.all(fb >= 0)
print(f"Q3 ✅  mel_filterbank shape={fb.shape}, weights non-negative")

# Q4: DCT orthogonality
from aurora.audio.mfcc import dct_ii
N_dct = 13
k_arr = np.arange(N_dct)
n_arr = np.arange(N_dct)
D = np.cos(np.pi * np.outer(k_arr, 2*n_arr+1) / (2*N_dct))
w0 = 1.0 / np.sqrt(N_dct)
wk = np.sqrt(2.0 / N_dct)
D[0] *= w0
D[1:] *= wk
err = np.max(np.abs(D @ D.T - np.eye(N_dct)))
assert err < 1e-12
print(f"Q4 ✅  DCT matrix orthogonality: max|D·Dᵀ-I|={err:.2e}")

# Q5: shape flow (aurora stft defaults to center=True: n_frames = 1 + N//hop)
sr, n_fft, hop, dur = 16000, 1024, 256, 1.0
N = int(sr * dur)
n_frames = 1 + N // hop
n_bins = n_fft // 2 + 1
n_mels, n_mfcc = 40, 13
assert n_frames == 63 and n_bins == 513
print(f"Q5 ✅  n_frames={n_frames}: power({n_frames},{n_bins})→Mel({n_frames},{n_mels})→MFCC({n_frames},{n_mfcc})")

# Q6: floating-point error — the order of magnitude of FFT error
x = np.random.randn(1024)
err_fft = np.max(np.abs(np.fft.fft(x) - np.fft.fft(x.copy())))
# Check that atol=1e-8 is reasonable (machine epsilon × N·log₂N)
machine_eps = np.finfo(float).eps
n_fft2 = 1024
error_bound = machine_eps * n_fft2 * np.log2(n_fft2)
print(f"Q6 ✅  machine epsilon={machine_eps:.2e}, N·log₂N error bound≈{error_bound:.2e} (1e-8 is safe)")
print("\n🎉 Six Audio Core interview questions passed! Your implementation stands on solid maths.")

In [ ]:
# ✏️ Self-review for this lesson (Audio Core complete)
l52_review = {
    "test_suite_green":        None,  # make test passes all 39 tests? True/False
    "fft_butterfly_formula":   None,  # can write X[k]=E[k]+W^k·O[k] on the whiteboard? True/False
    "mel_dct_pipeline":        None,  # can work out n_frames and every step's shape from the shape flow by hand? True/False
    "interview_ready":         None,  # can derive the complete 6-step formula chain without looking at code? True/False
    "whiteboard_passed":       None,  # finished all six interview questions on the whiteboard? True/False
}

unfilled = [k for k, v in l52_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l52_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work (polish these before any interview): {weak}')
else:
    print('✅ L52 fully cleared! Audio Core complete — on to L53: MFCCs illustrated')

---

→ **Next lesson** · [L53 · MFCCs illustrated](L53_visual_mfcc.ipynb)

> Next lesson covers **MFCCs illustrated**: waveform → spectrogram → mel spectrum → cepstral coefficients, drawn layer by layer.